# HCMAI Distributed Frame Extraction (Group)

Notebook này chạy Pipeline Extraction cục bộ trên Kaggle T4x2 hoặc P100.
Nó tải video từ S3 dựa trên danh sách `group-X.json`, chạy trích xuất frame,
và tự động upload kết quả lên thư mục tương ứng trên S3.

### Yêu cầu trước khi chạy:
1. Bật **Internet** và **GPU (T4 x2 hoặc P100)**.
2. Thêm token truy cập AWS S3 vào Kaggle Secrets: `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_DEFAULT_REGION`.
3. Upload (hoặc dán) file `group-X.json` của bạn và chỉ định đường dẫn ở cell dưới.

In [ ]:
# Đường dẫn tới file JSON phân chia nhóm (upload lên Kaggle Data hoặc paste nội dung vào 1 file)
GROUP_INVENTORY_PATH = "/kaggle/working/group-X.json"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("AWS_ACCESS_KEY_ID")
    os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("AWS_SECRET_ACCESS_KEY")
    os.environ["AWS_DEFAULT_REGION"] = secrets.get_secret("AWS_DEFAULT_REGION")
    print("✅ Loaded AWS credentials from Secrets.")
except Exception as e:
    print("⚠️ Không tìm thấy AWS credentials trong Kaggle Secrets. Vui lòng thêm vào!")

In [ ]:
%%bash
# ── System deps: FFmpeg (needed for extraction) ──
apt-get update -qq && apt-get install -y -qq --no-install-recommends ffmpeg libavutil-dev > /dev/null
echo "✅ Installed FFmpeg"

# ── Clone/Update repository ──
if ! [ -d /kaggle/working/HCMAI_2026/.git ]; then
    git clone https://github.com/khang1108/MLeCDanBGold /kaggle/working/HCMAI_2026
else
    git -C /kaggle/working/HCMAI_2026 pull --ff-only
fi
echo "✅ Repository is ready"

In [ ]:
%%bash
# Install dependencies (preprocessing)
pip install -q -e '/kaggle/working/HCMAI_2026[preprocessing,transcripts]'
pip install -q boto3
echo "✅ Python packages installed"

In [ ]:
import json
import boto3
import os

# 1. Đọc danh sách video từ Group JSON
with open(GROUP_INVENTORY_PATH, "r") as f:
    group_data = json.load(f)

bucket = group_data["bucket"]
videos = group_data["objects"]
print(f"Group ID: {group_data['group_id']}")
print(f"Cần tải {len(videos)} video từ bucket {bucket}")

# 2. Download Videos
download_dir = "/kaggle/working/videos"
os.makedirs(download_dir, exist_ok=True)

s3 = boto3.client("s3")
for idx, obj in enumerate(videos, 1):
    key = obj["key"]
    filename = os.path.basename(key)
    local_path = os.path.join(download_dir, filename)
    if not os.path.exists(local_path):
        print(f"[{idx}/{len(videos)}] Downloading {filename}...")
        s3.download_file(bucket, key, local_path)

print("✅ Download hoàn tất!")

In [ ]:
%%bash
# Chạy Group Pipeline (sẽ extract frames, artifacts và TỰ ĐỘNG UPLOAD lên S3 theo cơ chế group)
cd /kaggle/working/HCMAI_2026

python scripts/prepare_group_corpus.py \
  --videos /kaggle/working/videos \
  --inventory "$GROUP_INVENTORY_PATH" \
  --config configs/preparation.s3.yaml \
  --cleanup-raw  # (Tùy chọn: xóa video gốc sau khi xử lý xong để đỡ tốn dung lượng)
